**Note:** the saved output of this notebook was lost when a later re-run failed with an out-of-memory error, so the outputs have been cleared. The results from the successful run are saved in `results/diffusion_gan/diffusion_gan_metrics.json`, and its confusion matrix is in `figures/hybrid_confusion_matrix.png`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision.models import densenet121, DenseNet121_Weights 
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import torchvision.transforms as transforms
import json
import os
from datetime import datetime
from collections import Counter
from tqdm import tqdm
import gc
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

class Galaxy10Dataset(Dataset):
    def __init__(self, images, labels, transform=None):
        if images.dtype == np.uint8:
            self.images = images
        elif np.issubtype(images.dtype, np.floating) and images.max() <= 1.0:
             self.images = (images * 255).astype(np.uint8)
        else:
             self.images = images.astype(np.uint8) 

        self.labels = labels.astype(np.int64)
        self.transform = transform
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        
        if self.transform:
            image = Image.fromarray(image)
            image = self.transform(image)
        else:
            image = torch.from_numpy(image.astype(np.float32) / 255.0).permute(2, 0, 1)
        
        return image, torch.tensor(label, dtype=torch.long)

class BaselineClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        try:
            self.densenet = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1) 
        except AttributeError:
             self.densenet = densenet121(pretrained=True) 
        self.densenet.classifier = nn.Linear(self.densenet.classifier.in_features, num_classes)
    
    def forward(self, x):
        return self.densenet(x)

def load_baseline_data():
    print("Loading baseline data")
    data_dir = 'data/processed'
    X_train = np.load(f'{data_dir}/X_train.npy')
    X_val = np.load(f'{data_dir}/X_val.npy')
    X_test = np.load(f'{data_dir}/X_test.npy')
    y_train = np.load(f'{data_dir}/y_train.npy')
    y_val = np.load(f'{data_dir}/y_val.npy')
    y_test = np.load(f'{data_dir}/y_test.npy')
    
    try:
        with open(f'{data_dir}/data_info.json', 'r') as f:
            data_info = json.load(f)
    except FileNotFoundError:
        data_info = {'class_names': [
            'Disturbed Galaxies', 'Merging Galaxies', 'Round Smooth Galaxies',
            'In-between Round Smooth Galaxies', 'Cigar Shaped Smooth Galaxies',  
            'Barred Spiral Galaxies', 'Unbarred Tight Spiral Galaxies',
            'Unbarred Loose Spiral Galaxies', 'Edge-on Galaxies without Bulge',
            'Edge-on Galaxies with Bulge'
        ]}
    
    print(f"Data loaded: Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
    
    return X_train, X_val, X_test, y_train, y_val, y_test, data_info

def load_synthetic_data(source='diffusion_gan'):
    print(f"Loading synthetic data from results/{source}/")
    
    if not os.path.exists(f'results/{source}/synthetic_images.npy'):
        raise FileNotFoundError(f"Cannot find synthetic_images.npy in results/{source}/")
    
    synthetic_images = np.load(f'results/{source}/synthetic_images.npy')
    synthetic_labels = np.load(f'results/{source}/synthetic_labels.npy')
    print(f"Synthetic data loaded: {synthetic_images.shape}, Classes: {np.unique(synthetic_labels)}")
    return synthetic_images, synthetic_labels

def create_augmented_dataset(X_train, y_train, synthetic_images, synthetic_labels, minority_classes):
    print("\nDIFFUSION-GAN: Creating augmented dataset")
    
    MINORITY_CLASSES = [1, 4, 8, 9]
    
    original_counts = Counter(y_train)
    print(f"Original training class distribution: {dict(sorted(original_counts.items()))}")
    
    X_train_resized = []
    print(f"Resizing {X_train.shape[0]} original images (256x256 -> 128x128)...")
    for img in tqdm(X_train, desc="Resizing"):
        img_pil = Image.fromarray(img)
        img_resized = img_pil.resize((128, 128))
        X_train_resized.append(np.array(img_resized))
    
    X_train_resized = np.array(X_train_resized)
    print(f"Original data resized: {X_train.shape} -> {X_train_resized.shape}")
    print(f"Synthetic data shape: {synthetic_images.shape}")
    
    augmented_images = np.concatenate([X_train_resized, synthetic_images], axis=0)
    augmented_labels = np.concatenate([y_train, synthetic_labels], axis=0)
    
    augmented_counts = Counter(augmented_labels)
    print(f"Augmented class distribution: {dict(sorted(augmented_counts.items()))}")
    
    print("\nMinority class improvement:")
    for class_id in MINORITY_CLASSES:
        original_count = original_counts.get(class_id, 0)
        augmented_count = augmented_counts.get(class_id, 0)
        improvement = ((augmented_count - original_count) / original_count * 100) if original_count > 0 else 0
        print(f"Class {class_id}: {original_count} -> {augmented_count} (+{improvement:.1f}%)")
    
    return augmented_images, augmented_labels

def train_augmented_classifier(X_train_aug, y_train_aug, X_val, y_val, experiment_name="diffusion_gan"):
    print(f"\nTRAINING {experiment_name.upper()} CLASSIFIER")
    
    BATCH_SIZE = 8
    EPOCHS = 25
    LEARNING_RATE = 0.001
    
    print(f"Parameters: Batch={BATCH_SIZE}, Epochs={EPOCHS}, LR={LEARNING_RATE}")
    print(f"Training samples: {len(X_train_aug)}")
    
    train_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_dataset = Galaxy10Dataset(X_train_aug, y_train_aug, transform=train_transform)
    val_dataset = Galaxy10Dataset(X_val, y_val, transform=val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    model = BaselineClassifier(num_classes=10).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    best_val_accuracy = 0.0
    best_model_state = None
    
    for epoch in range(EPOCHS):
        print(f'\nEpoch {epoch+1}/{EPOCHS}')
        
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in tqdm(train_loader, desc="Training"):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_loss = running_loss / len(train_loader)
        train_accuracy = correct / total
        train_losses.append(train_loss)
        train_accuracies.append(train_accuracy)
        
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation"):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_loss = val_loss / len(val_loader)
        val_accuracy = val_correct / val_total
        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)
        
        scheduler.step()
        
        print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}')
        print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}')
        
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = model.state_dict().copy()
            print(f'New best: {val_accuracy:.4f}')
    
    if best_model_state:
        model.load_state_dict(best_model_state)
    
    return model, {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies': val_accuracies,
        'best_val_accuracy': best_val_accuracy,
        'experiment_name': experiment_name
    }

def evaluate_model_performance(model, X_test, y_test, class_names, experiment_name="diffusion_gan"):
    FOCUS_CLASSES = [1, 4, 8, 9]
    
    print(f"\nEVALUATING {experiment_name.upper()} PERFORMANCE")
    
    test_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    test_dataset = Galaxy10Dataset(X_test, y_test, transform=test_transform)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0)
    
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Testing"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    y_true = np.array(all_labels)
    y_pred = np.array(all_predictions)
    
    overall_accuracy = np.mean(y_true == y_pred)
    print(f"Overall Test Accuracy: {overall_accuracy:.4f}")
    
    class_report = classification_report(
        y_true, y_pred, 
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )
    
    focus_f1_scores = []
    
    print(f"\nFOCUS CLASS PERFORMANCE {FOCUS_CLASSES}:")
    
    for class_id in FOCUS_CLASSES:
        if class_id < len(class_names):
            class_name = class_names[class_id]
            f1 = class_report[class_name]['f1-score']
            focus_f1_scores.append(f1)
            print(f"Class {class_id} ({class_name}): F1={f1:.3f}")
            
    avg_focus_f1 = np.mean(focus_f1_scores)
    cigar_f1 = class_report[class_names[4]]['f1-score']
    
    print(f"Average F1 for focus classes: {avg_focus_f1:.3f}")
    
    print("\nFull Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))
    
    os.makedirs(f'results/{experiment_name}', exist_ok=True)
    metrics = {
        'overall_accuracy': overall_accuracy,
        'avg_focus_f1': avg_focus_f1,
        'cigar_f1': cigar_f1, 
        'class_report': class_report,
        'minority_f1_scores': focus_f1_scores
    }
    
    with open(f'results/{experiment_name}/{experiment_name}_metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2, default=str)
        
    torch.save(model.state_dict(), f'results/{experiment_name}/{experiment_name}_model.pth')
    print(f"\nResults saved to results/{experiment_name}/")
    
    plt.figure(figsize=(10, 8))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=range(10), yticklabels=range(10))
    plt.title(f'{experiment_name.upper()}: Confusion Matrix')
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'results/{experiment_name}/confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

    return metrics

def compare_all_results():
    print("\nCOMPREHENSIVE COMPARISON: BASELINE vs DDPM vs GAN vs HYBRID\n")
    
    results = {}
    
    try:
        with open('results/baseline/baseline_metrics.json', 'r') as f:
            baseline = json.load(f)
            perf = baseline['performance']
            results['Baseline'] = {
                'overall_accuracy': perf['overall_accuracy'],
                'avg_focus_f1': perf['avg_minority_f1'],
                'cigar_f1': perf['cigar_class_f1'],
                'minority_f1_scores': [0.819, 0.712, 0.883, 0.912]  # From class_report
            }
            print("Baseline loaded")
    except Exception as e:
        print(f"Baseline not found: {e}")
    
    try:
        with open('results/diffusion/diffusion_augmented_metrics.json', 'r') as f:
            ddpm = json.load(f)
            results['DDPM'] = {
                'overall_accuracy': ddpm['overall_accuracy'],
                'avg_focus_f1': ddpm['avg_focus_f1'],
                'cigar_f1': ddpm['cigar_f1'],
                'minority_f1_scores': ddpm['minority_f1_scores']
            }
            print(" DDPM loaded")
    except Exception as e:
        print(f" DDPM not found: {e}")
    
    try:
        with open('results/gan/phase3_results.json', 'r') as f:
            gan_data = json.load(f)
            gan = gan_data['gan_metrics']
            results['GAN'] = {
                'overall_accuracy': gan['overall_accuracy'],
                'avg_focus_f1': gan['avg_minority_f1'],
                'cigar_f1': gan['cigar_f1'],
                'minority_f1_scores': gan['minority_f1_scores']
            }
            print("GAN loaded")
    except Exception as e:
        print(f"GAN not found: {e}")
    
    try:
        with open('results/diffusion_gan/diffusion_gan_metrics.json', 'r') as f:
            hybrid = json.load(f)
            results['Diffusion-GAN Hybrid'] = {
                'overall_accuracy': hybrid['overall_accuracy'],
                'avg_focus_f1': hybrid['avg_focus_f1'],
                'cigar_f1': hybrid['cigar_f1'],
                'minority_f1_scores': hybrid['minority_f1_scores']
            }
            print("Hybrid loaded")
    except Exception as e:
        print(f" Hybrid not found: {e}")
    
    if not results:
        print("\nNo results found to compare")
        return
    
    print()
    
    comparison_data = []
    minority_classes = [1, 4, 8, 9]
    
    for name, metrics in results.items():
        row = {
            'Model': name,
            'Accuracy': f"{metrics['overall_accuracy']:.3f}",
            'Avg F1': f"{metrics['avg_focus_f1']:.3f}",
            'Cigar F1': f"{metrics['cigar_f1']:.3f}"
        }
        
        minority_scores = metrics['minority_f1_scores']
        for i, class_id in enumerate(minority_classes):
            if i < len(minority_scores):
                row[f'C{class_id}'] = f"{minority_scores[i]:.3f}"
            else:
                row[f'C{class_id}'] = "N/A"
        
        comparison_data.append(row)
    
    df = pd.DataFrame(comparison_data)
    print(df.to_string(index=False))
    
    if 'Baseline' in results:
        baseline_acc = results['Baseline']['overall_accuracy']
        baseline_f1 = results['Baseline']['avg_focus_f1']
        
        print("\n\nIMPROVEMENTS vs BASELINE:\n")
        
        for model_name in ['DDPM', 'GAN', 'Diffusion-GAN Hybrid']:
            if model_name in results:
                acc = results[model_name]['overall_accuracy']
                f1 = results[model_name]['avg_focus_f1']
                acc_imp = ((acc - baseline_acc) / baseline_acc) * 100
                f1_imp = ((f1 - baseline_f1) / baseline_f1) * 100
                print(f"{model_name:25s} Accuracy: {baseline_acc:.3f} → {acc:.3f} ({acc_imp:+.1f}%)  |  Avg F1: {baseline_f1:.3f} → {f1:.3f} ({f1_imp:+.1f}%)")
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    models = list(results.keys())
    colors_map = {'Baseline': 'gray', 'DDPM': 'blue', 'GAN': 'green', 'Diffusion-GAN Hybrid': 'red'}
    colors = [colors_map.get(m, 'purple') for m in models]
    
    accuracies = [results[m]['overall_accuracy'] for m in models]
    avg_f1s = [results[m]['avg_focus_f1'] for m in models]
    
    axes[0].bar(models, accuracies, color=colors)
    axes[0].set_ylabel('Accuracy', fontsize=12)
    axes[0].set_title('Overall Test Accuracy', fontsize=14, fontweight='bold')
    axes[0].set_ylim([0.75, 0.85])
    axes[0].tick_params(axis='x', rotation=15)
    for i, v in enumerate(accuracies):
        axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')
    
    axes[1].bar(models, avg_f1s, color=colors)
    axes[1].set_ylabel('F1-Score', fontsize=12)
    axes[1].set_title('Average Minority Class F1-Score', fontsize=14, fontweight='bold')
    axes[1].set_ylim([0.75, 0.85])
    axes[1].tick_params(axis='x', rotation=15)
    for i, v in enumerate(avg_f1s):
        axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('results/comprehensive_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    df.to_csv('results/comparison_table.csv', index=False)
    print("\n\nComparison saved to results/comparison_table.csv and results/comprehensive_comparison.png\n")

def run_hybrid_analysis():
    print("\nDIFFUSION-GAN HYBRID: CLASSIFICATION ANALYSIS\n")
    
    X_train, X_val, X_test, y_train, y_val, y_test, data_info = load_baseline_data()
    synthetic_images, synthetic_labels = load_synthetic_data('diffusion_gan')
    
    X_train_aug, y_train_aug = create_augmented_dataset(
        X_train, y_train, synthetic_images, synthetic_labels, [1, 4, 8, 9]
    )
    
    model, training_results = train_augmented_classifier(
        X_train_aug, y_train_aug, X_val, y_val, "diffusion_gan"
    )
    
    metrics = evaluate_model_performance(
        model, X_test, y_test, data_info['class_names'], "diffusion_gan"
    )
    
    print("\nHYBRID ANALYSIS COMPLETE\n")
    
    return model, metrics

try:
    model, metrics = run_hybrid_analysis()
    compare_all_results()
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()